# Male Loneliness Project — Reddit + Google Trends

**Run the cells top to bottom.** Every data cell saves its results to `project_outputs/` (Excel + CSV) even if it is interrupted, and Reddit results are always sorted by popularity (score, then number of comments).

| Part | What it does | Period | Data source |
|---|---|---|---|
| 0 | Install + settings + shared helpers | – | – |
| 1A | Monthly post/comment volume of each subreddit (time series) | 2017–present, monthly | Arctic Shift API |
| 1B | All loneliness posts per subreddit per **year**, ranked by score | 2020–present, yearly | Arctic Shift API |
| 1C | Top comments on each year's most popular posts | 2020–present, yearly | Arctic Shift API |
| 1E | Likely-Canadian users + subreddit similarity scores (538 "subreddit algebra" idea) | 2020–present | Arctic Shift API |
| 1F | Most frequent / most distinctive terms per year | 2020–present, yearly | your collected data |
| 2A–2D | Google Trends for **Canada only**: over time, related queries, comparisons, by province | 2017–present | pytrends |

(Section letters match earlier versions; 1D, the all-of-Reddit PullPush search, was removed.)

**Start with `TEST_MODE = True`** (in the settings cell). Reddit sections then only pull the last 3 months, so you can confirm everything works in a few minutes. Set it to `False` for the full run, which takes hours — run it overnight or one subreddit at a time.

All Reddit data come from Arctic Shift, a free archive of public Reddit data. It lets us filter by subreddit and date, gives ready-made monthly counts, and can tell us which subreddits a user is active in (needed for the Canadian analysis).

## Part 0 — Setup

In [1]:
# Run once per environment (safe to re-run). %pip installs into the kernel you're actually using.
%pip install -q requests pandas numpy matplotlib openpyxl pytrends scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [1]:
# ============================================================
# SETTINGS + SHARED HELPERS  (run this before any other cell)
# ============================================================
import os, re, time, json, random, hashlib, warnings
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

warnings.filterwarnings("ignore", category=FutureWarning)   # pytrends triggers harmless pandas FutureWarnings

# ---------- Study window ----------
START_DATE = "2017-01-01"
END_DATE = pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d")   # always "today", no hard-coded end date

# ---------- Test mode ----------
# True  = only the last TEST_MONTHS months and small samples (a few minutes) -> check everything works
# False = full 2017-present run (long; leave it running)
TEST_MODE = False
TEST_MONTHS = 3

# ---------- Where results go ----------
OUTPUT_DIR = "project_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------- Reddit: what to search ----------
# Keyword search is WORD based, so "lone" will NOT match "lonely" (that's why your old r/AskMen cell found little).
# List every word form you want.
LONELINESS_KEYWORDS = ["lonely", "loneliness", "alone", "isolated", "isolation",
                       "no friends", "friendless", "disconnected"]

# subreddit -> keywords.  None = the whole subreddit is on-topic, so take ALL posts.
# Notes: subreddit names have no spaces (r/ForeverAlone).  r/MGTOW was banned by Reddit in Aug 2021,
# so its data stops there (months after that will correctly show 0).
TARGET_SUBREDDITS = {
    "lonely":       None,
    "ForeverAlone": None,
    "AskMen":       LONELINESS_KEYWORDS,
    "MGTOW":        None,
    # ideas to add: "MensLib": LONELINESS_KEYWORDS, "socialanxiety": None, "MensMentalHealth": None,
}

# Canadian local subreddits, used to infer which users are probably Canadian (Part 1E)
CANADIAN_SUBREDDITS = [
    "canada", "onguardforthee", "CanadaPolitics", "AskACanadian", "PersonalFinanceCanada", "CanadaHousing2",
    "ontario", "toronto", "askTO", "ottawa", "mississauga", "hamilton", "kitchener", "londonontario",
    "britishcolumbia", "vancouver", "VictoriaBC", "alberta", "Calgary", "Edmonton",
    "saskatchewan", "saskatoon", "regina", "manitoba", "Winnipeg",
    "quebec", "montreal", "halifax", "NovaScotia", "newbrunswickcanada", "PEI", "newfoundland",
    "yukon", "nwt", "nunavut",
]

# Usernames are personal data. Exported files only contain a salted hash of each username.
# Change this to any private string and don't share it.
AUTHOR_SALT = "change-me-to-a-private-string"

print(f"Study window: {START_DATE} -> {END_DATE} | TEST_MODE={TEST_MODE} | outputs -> {os.path.abspath(OUTPUT_DIR)}")


# =================== generic helpers ===================
def to_timestamp(v):
    """Epoch seconds / epoch ms / ISO string -> tz-aware UTC Timestamp (NaT if impossible)."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return pd.NaT
    try:
        if isinstance(v, str) and not re.fullmatch(r"-?\d+(\.\d+)?", v.strip()):
            return pd.to_datetime(v, utc=True)
        x = float(v)
        return pd.to_datetime(x, unit="ms" if x > 1e11 else "s", utc=True)
    except Exception:
        return pd.NaT

def to_epoch(v):
    ts = to_timestamp(v)
    return None if pd.isna(ts) else int(ts.timestamp())

def month_label(ts):
    """Timestamp -> 'YYYY-MM'.  (+12h guards against servers that report month starts as 23:00 the day before.)"""
    if pd.isna(ts):
        return None
    return (ts + pd.Timedelta(hours=12)).strftime("%Y-%m")

def month_windows(start=START_DATE, end=END_DATE):
    """List of (label, after_epoch, before_epoch) for each calendar month (UTC)."""
    periods = pd.period_range(start=start, end=end, freq="M")
    if TEST_MODE:
        periods = periods[-TEST_MONTHS:]
    out = []
    for p in periods:
        a = int(p.start_time.tz_localize("UTC").timestamp())
        b = int(p.end_time.tz_localize("UTC").timestamp()) + 1
        out.append((str(p), a, b))
    return out

CURRENT_MONTH = pd.Timestamp.now(tz="UTC").strftime("%Y-%m")

def hash_author(name):
    if not isinstance(name, str) or name in ("", "[deleted]", "AutoModerator"):
        return name
    return hashlib.sha256((AUTHOR_SALT + name.lower()).encode("utf-8")).hexdigest()[:12]

def _clean_for_export(df):
    if df is None or len(df) == 0:
        return pd.DataFrame({"note": ["no rows"]})
    df = df.copy()
    for c in df.columns:
        if isinstance(df[c].dtype, pd.DatetimeTZDtype):            # Excel can't store time zones
            df[c] = df[c].dt.tz_convert("UTC").dt.tz_localize(None)
        elif df[c].dtype == object or pd.api.types.is_string_dtype(df[c]):
            def fix(v):
                if isinstance(v, (list, dict, tuple, set)):
                    v = json.dumps(list(v) if isinstance(v, (set, tuple)) else v, ensure_ascii=False)
                if isinstance(v, str):
                    v = ILLEGAL_CHARACTERS_RE.sub("", v)[:32000]    # control chars crash openpyxl; Excel cell max 32,767
                return v
            df[c] = df[c].map(fix)
    return df

def save_tables(sheets, basename):
    """Write {sheet_name: DataFrame} to one .xlsx plus one UTF-8 .csv per sheet."""
    xlsx = os.path.join(OUTPUT_DIR, f"{basename}.xlsx")
    with pd.ExcelWriter(xlsx, engine="openpyxl") as w:
        for name, df in sheets.items():
            clean = _clean_for_export(df)
            clean.to_excel(w, sheet_name=name[:31], index=False)
            clean.to_csv(os.path.join(OUTPUT_DIR, f"{basename}__{re.sub(r'[^A-Za-z0-9]+', '_', name)}.csv"),
                         index=False, encoding="utf-8-sig")   # utf-8-sig = opens cleanly in Excel on Windows
    print(f"Saved: {os.path.abspath(xlsx)}  (+ CSVs)")

def load_reddit_posts():
    """Use posts from Part 1B (in memory, or reloaded from disk)."""
    if "reddit_posts" in globals() and isinstance(globals()["reddit_posts"], pd.DataFrame) and len(globals()["reddit_posts"]):
        return globals()["reddit_posts"]
    path = os.path.join(OUTPUT_DIR, "reddit_posts_raw.pkl")
    if os.path.exists(path):
        return pd.read_pickle(path)
    raise RuntimeError("No Reddit posts yet - run Part 1B first.")


# =================== Arctic Shift API client ===================
ARCTIC = "https://arctic-shift.photon-reddit.com"
SESSION = requests.Session()
SESSION.headers["User-Agent"] = "mens-loneliness-research/1.0 (university course project)"
ARCTIC_MIN_INTERVAL = 1.0      # seconds between requests (free volunteer service - be polite)
_last_call = {"arctic": 0.0, "pullpush": 0.0}

def _pace(key, interval):
    wait = interval - (time.time() - _last_call[key])
    if wait > 0:
        time.sleep(wait)
    _last_call[key] = time.time()


def arctic_get(endpoint, params, max_retries=5):
    """GET an Arctic Shift endpoint. Returns (data, status) where status is 'ok' or an error description."""
    global ARCTIC_MIN_INTERVAL
    timeouts = 0
    for attempt in range(max_retries):
        _pace("arctic", ARCTIC_MIN_INTERVAL)
        try:
            r = SESSION.get(ARCTIC + endpoint, params=params, timeout=120)
        except requests.exceptions.RequestException as e:
            print(f"    connection problem ({type(e).__name__}); retrying...")
            time.sleep(5 * (attempt + 1))
            continue
        if r.status_code == 429:
            ARCTIC_MIN_INTERVAL = min(ARCTIC_MIN_INTERVAL * 1.5, 20)   # back off after each limit
            reset = r.headers.get("X-RateLimit-Reset")
            wait = float(reset) + 1 if reset else 15 * 2 ** attempt
            print(f"    rate limited - waiting {wait:.0f}s (attempt {attempt + 1}/{max_retries})")
            time.sleep(min(wait, 900))
            continue
        if r.status_code >= 500:
            time.sleep(5 * (attempt + 1))
            continue
        try:
            payload = r.json()
        except ValueError:
            return None, f"bad_response_http_{r.status_code}"
        err = payload.get("error") if isinstance(payload, dict) else None
        if r.status_code != 200 or err:
            msg = str(err or f"http_{r.status_code}")
            if "timed out" in msg.lower() and timeouts < 1:     # DB sometimes needs a warm-up; retry once
                timeouts += 1
                time.sleep(3)
                continue
            return None, msg[:200]
        ARCTIC_MIN_INTERVAL = max(1.0, ARCTIC_MIN_INTERVAL * 0.95)    # slowly speed back up
        return (payload.get("data") if isinstance(payload, dict) else payload), "ok"
    return None, "max_retries_exceeded"

def arctic_fetch_all(endpoint, params, after, before, max_items=10000):
    """Page through every result in [after, before) using the created_utc cursor. Returns (items, status)."""
    items, seen = [], set()
    cursor, page_limit = after, "auto"            # "auto" = 100-1000 rows per request
    while True:
        q = dict(params, after=cursor, before=before, sort="asc", limit=page_limit)
        data, status = arctic_get(endpoint, q)
        if data is None:
            if page_limit == "auto":              # big pages can time out -> fall back to 100 per page
                page_limit = 100
                continue
            return items, status
        new = [d for d in data if d.get("id") not in seen]
        if not new:
            break
        seen.update(d.get("id") for d in new)
        items.extend(new)
        if len(items) >= max_items:
            return items, "capped"
        last = max((to_epoch(d.get("created_utc")) or cursor) for d in new)
        if last >= before or (page_limit == 100 and len(data) < 100):
            break
        cursor = last
    return items, "ok"

def status_ok(s):
    return isinstance(s, str) and (s.startswith("ok") or s == "capped")

print("Helpers loaded.")

Study window: 2017-01-01 -> 2026-10-09 | TEST_MODE=False | outputs -> C:\Users\vesta\Downloads\project_outputs
Helpers loaded.


## Part 1 — Reddit

### 1B. Loneliness posts per subreddit per year (2020–present) — ranked by popularity
For every subreddit and year it collects **all** matching posts (whole subreddit for r/lonely, r/ForeverAlone and r/MGTOW; keyword matches for r/AskMen), then ranks them by score within each year.

Why each year is fetched in monthly pieces: the archive only returns posts in date order and can't sort by score, so every post in the year has to be collected before ranking. A whole year in one request times out or hits size caps (which would silently keep only January–March). Monthly pieces avoid that; all results are grouped, ranked and summarised **by year**.

Outputs: the top posts per year, every matching post (sorted by score within each year), a yearly summary (your keyword-discourse trend), and a diagnostics sheet that separates real zeros from failed requests. The current year is partial, and scores of posts under ~36 hours old are provisional.

In [ ]:
REDDIT_START = "2020-01-01"     # start of the Reddit sections 1B, 1C, 1E, 1F
TOP_N_PER_YEAR = 100            # rows in the "Top posts" sheet per subreddit-year
MAX_POSTS_PER_CHUNK = 10000     # safety cap per subreddit-month-keyword piece
POST_FIELDS = "id,author,created_utc,subreddit,title,selftext,score,num_comments,link_flair_text,url,over_18"
KW_REGEX = re.compile(r"\b(" + "|".join(re.escape(k) for k in LONELINESS_KEYWORDS) + r")\b", re.I)
CURRENT_YEAR = pd.Timestamp.now(tz="UTC").year

def fetch_posts(sub, a, b, keyword=None, depth=0):
    params = {"subreddit": sub, "fields": POST_FIELDS}
    if keyword:
        params["query"] = keyword            # searches title + body
    items, status = arctic_fetch_all("/api/posts/search", params, a, b, max_items=MAX_POSTS_PER_CHUNK)
    # keyword search on big subreddits can time out -> split the piece into 4 smaller windows and retry
    if not status_ok(status) and depth == 0 and (b - a) > 3 * 86400:
        edges = np.linspace(a, b, 5).astype(int)
        items, sts = [], []
        for x, y in zip(edges[:-1], edges[1:]):
            it, st = fetch_posts(sub, int(x), int(y), keyword, depth=1)
            items += it; sts.append(st)
        status = "ok(split)" if all(status_ok(s) for s in sts) else "partial: " + " / ".join(sorted(set(sts)))
    return items, status

def post_record(d, sub, keyword):
    ts = to_timestamp(d.get("created_utc"))
    title, body = d.get("title") or "", d.get("selftext") or ""
    pid = d.get("id", "")
    return {
        "subreddit": d.get("subreddit") or sub, "year": ts.year if pd.notna(ts) else None,
        "date": ts, "score": d.get("score", 0) or 0, "num_comments": d.get("num_comments", 0) or 0,
        "title": title, "selftext": body,
        "keyword_hits": "; ".join(sorted({m.lower() for m in KW_REGEX.findall(f"{title} {body}")})),
        "search_keyword": keyword or "(all posts)",
        "flair": d.get("link_flair_text"), "post_id": pid,
        "permalink": f"https://www.reddit.com/r/{d.get('subreddit') or sub}/comments/{pid}/",
        "author": d.get("author"),
        "body_removed": body in ("[removed]", "[deleted]"),
    }

def build_post_tables(records, diagnostics):
    diag = pd.DataFrame(diagnostics)
    df = pd.DataFrame(records)
    if df.empty:
        return df, df, df, diag
    kw = df.groupby("post_id")["search_keyword"].agg(lambda s: "; ".join(sorted(set(s))))
    df = df.drop_duplicates("post_id").drop(columns="search_keyword")
    df["matched_search_keywords"] = df["post_id"].map(kw)
    df["score"] = pd.to_numeric(df["score"], errors="coerce").fillna(0).astype(int)
    df["num_comments"] = pd.to_numeric(df["num_comments"], errors="coerce").fillna(0).astype(int)
    df = df.dropna(subset=["year"]).astype({"year": int})
    df = df.sort_values(["subreddit", "year", "score", "num_comments"], ascending=[True, True, False, False])
    df["rank_in_year"] = df.groupby(["subreddit", "year"]).cumcount() + 1
    df["partial_year"] = df["year"] == CURRENT_YEAR
    top = df[df["rank_in_year"] <= TOP_N_PER_YEAR]

    yearly = (df.groupby(["subreddit", "year"])
                .agg(n_posts=("post_id", "count"),
                     n_posts_with_keyword_in_text=("keyword_hits", lambda s: (s != "").sum()),
                     unique_authors=("author", "nunique"),
                     mean_score=("score", "mean"), median_score=("score", "median"), max_score=("score", "max"),
                     total_comments=("num_comments", "sum"))
                .reset_index())
    # add the request status per subreddit-year, and explicit zero rows for years with no posts
    if not diag.empty:
        d = (diag.assign(failed=~diag["status"].map(status_ok))
                 .groupby(["subreddit", "year"])
                 .agg(pieces_searched=("status", "size"), pieces_failed=("failed", "sum")).reset_index())
        yearly = d.merge(yearly, on=["subreddit", "year"], how="left")
        yearly["n_posts"] = yearly["n_posts"].fillna(0).astype(int)
    yearly["partial_year"] = yearly["year"] == CURRENT_YEAR
    return df, top, yearly.sort_values(["subreddit", "year"]), diag

def export_posts(df, top, yearly, diag):
    def public(x):
        if x.empty: return x
        x = x.copy(); x["author_hash"] = x["author"].map(hash_author)
        return x.drop(columns=["author"])
    save_tables({"Top posts per year": public(top), "All matching posts": public(df),
                 "Yearly summary": yearly, "Diagnostics": diag}, "reddit_1B_posts_by_popularity")
    if not df.empty:
        df.to_pickle(os.path.join(OUTPUT_DIR, "reddit_posts_raw.pkl"))   # raw usernames; local use only, don't share

# ---------------- run ----------------
plan = {}
for label, a, b in month_windows(start=REDDIT_START):          # monthly pieces, grouped by year
    plan.setdefault(int(label[:4]), []).append((label, a, b))
print(f"{len(TARGET_SUBREDDITS)} subreddits x years {min(plan)}-{max(plan)}"
      + (" (TEST_MODE: last few months only)" if TEST_MODE else ""))

records, diagnostics = [], []
try:
    for sub, kws in TARGET_SUBREDDITS.items():
        for year, pieces in plan.items():
            n_before, fails = len(records), 0
            for label, a, b in pieces:
                for kw in (kws or [None]):
                    items, status = fetch_posts(sub, a, b, kw)
                    records += [post_record(d, sub, kw) for d in items]
                    fails += not status_ok(status)
                    diagnostics.append({"subreddit": sub, "year": year, "month_piece": label,
                                        "keyword": kw or "(all posts)", "status": status, "n_returned": len(items)})
            print(f"r/{sub} {year}: {len({r['post_id'] for r in records[n_before:]}):,} unique posts collected"
                  + (f"  ({fails} failed pieces - see Diagnostics)" if fails else ""))
except KeyboardInterrupt:
    print("\nInterrupted - saving everything collected so far...")
finally:
    reddit_posts, top_posts, yearly_posts, posts_diag = build_post_tables(records, diagnostics)
    export_posts(reddit_posts, top_posts, yearly_posts, posts_diag)
    if len(posts_diag):
        print(f"Unique posts: {len(reddit_posts):,} | failed/partial pieces: {(~posts_diag['status'].map(status_ok)).sum()} of {len(posts_diag)}")

# ---- plot the yearly keyword-discourse trend ----
if len(yearly_posts):
    subs = yearly_posts["subreddit"].unique()
    fig, axes = plt.subplots(len(subs), 1, figsize=(10, 3 * len(subs)), squeeze=False)
    for ax, sub in zip(axes[:, 0], subs):
        g = yearly_posts[yearly_posts["subreddit"] == sub]
        colors = ["#bbbbbb" if p else "#3b6ea8" for p in g["partial_year"]]
        ax.bar(g["year"].astype(str), g["n_posts"], color=colors)
        label = "posts with loneliness keywords" if TARGET_SUBREDDITS.get(sub) else "all posts"
        ax.set_title(f"r/{sub} - {label} per year (grey = current, partial year)"); ax.grid(axis="y", alpha=.3)
    plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, "reddit_1B_yearly_posts.png"), dpi=150); plt.show()

### 1E. Likely-Canadian users and subreddit similarity scores

**What the "538 subreddit analyser" is:** you're most likely thinking of FiveThirtyEight's (538) *"subreddit algebra"* tool (Trevor Martin, 2017). It scored how similar two subreddits are by **commenter overlap** — two subreddits are similar if many of the same people are active in both. It was built on 2015–2016 data and isn't a live tool you can point at new data, so this cell reproduces the idea on current data.

**How Reddit location works:** Reddit has no location field. The standard workaround is to treat users who are active in Canadian local subreddits (r/toronto, r/ontario, r/vancouver, r/canada…) as *probably* Canadian. This cell:

1. Takes the most active authors in your target subreddits (from 1B), plus a sample of active authors from Canadian subreddits.
2. Asks Arctic Shift which subreddits each of those users has posted/commented in since 2020.
3. Flags a user as **likely Canadian** if they have ≥ `MIN_CANADIAN_INTERACTIONS` interactions in Canadian subreddits.
4. Computes **similarity scores** between subreddits from the shared users:
   - **cosine similarity** (0–1): how similar the two subreddits' user bases are, weighting by how active people are in each;
   - **lift**: how many times more often users overlap than chance would predict (1 = chance, 3 = three times more than chance).
5. Lists the subreddits most related to each target subreddit (a map of the surrounding discourse ecosystem — e.g. which mental-health or dating subs r/lonely users also use), and the share of each target subreddit's users who look Canadian.
6. Exports the 1B posts written by likely-Canadian users, ranked by score within each year.

**Caveats to state in your write-up:** this is an inference, not ground truth — many Canadians never post in local subs, and some non-Canadians do. Scores come from a sample of users, so treat them as indicative. Only exported hashed usernames; check whether your course/REB requires anything further for user-level Reddit data.

In [ ]:
REDDIT_START = globals().get("REDDIT_START", "2020-01-01")
AUTHORS_PER_TARGET_SUB   = 30 if TEST_MODE else 200   # most active authors from each target subreddit
CANADA_SAMPLE_SUBS       = ["canada", "ontario", "toronto", "ottawa", "vancouver", "britishcolumbia",
                            "alberta", "Calgary", "Edmonton", "Winnipeg", "montreal", "halifax"]
AUTHORS_PER_CANADA_SUB   = 8 if TEST_MODE else 40
CANADA_SAMPLE_DAYS       = 14 if TEST_MODE else 60    # recent window used to find active Canadian-sub users
MIN_CANADIAN_INTERACTIONS = 3
MIN_USERS_FOR_SIMILARITY = 5                          # ignore subreddits shared by fewer sampled users
SKIP_AUTHORS = {"[deleted]", "AutoModerator", "", None}

canada_lower = {s.lower() for s in CANADIAN_SUBREDDITS}
posts = load_reddit_posts()

# ---- 1. target-subreddit authors: most posts first, ties broken by total score ----
author_source = {}
for sub, g in posts[~posts["author"].isin(SKIP_AUTHORS)].groupby("subreddit"):
    ranked = (g.groupby("author").agg(n=("post_id", "count"), s=("score", "sum"))
                .sort_values(["n", "s"], ascending=False).head(AUTHORS_PER_TARGET_SUB))
    for a in ranked.index:
        author_source.setdefault(a, f"target:r/{sub}")

# ---- 2. Canadian-subreddit authors ----
now = int(pd.Timestamp.now(tz="UTC").timestamp())
for sub in CANADA_SAMPLE_SUBS:
    items, st = arctic_fetch_all("/api/posts/search", {"subreddit": sub, "fields": "id,author,created_utc"},
                                 now - CANADA_SAMPLE_DAYS * 86400, now, max_items=2000)
    counts = pd.Series([d.get("author") for d in items if d.get("author") not in SKIP_AUTHORS]).value_counts()
    for a in counts.head(AUTHORS_PER_CANADA_SUB).index:
        author_source.setdefault(a, f"canada:r/{sub}")
    print(f"r/{sub}: {len(items)} recent posts, {min(len(counts), AUTHORS_PER_CANADA_SUB)} authors sampled ({st})")

# ---- 3. which subreddits is each author active in? ----
def parse_counts(data):
    out = {}
    rows = data.items() if isinstance(data, dict) else (data or [])
    for row in rows:
        if isinstance(row, dict):
            name = row.get("subreddit") or row.get("key") or next((v for v in row.values() if isinstance(v, str)), None)
            cnt = row.get("count", row.get("doc_count", row.get("weight")))
            if cnt is None:
                cnt = next((v for k, v in row.items() if k != "subreddit" and pd.notna(pd.to_numeric(v, errors="coerce"))), 1)
        elif isinstance(row, (list, tuple)) and len(row) >= 2:
            name, cnt = row[0], row[1]
        else:
            continue
        if name:
            out[str(name).lower()] = out.get(str(name).lower(), 0) + float(pd.to_numeric(cnt, errors="coerce") or 0)
    return out

authors = list(author_source)
print(f"\nLooking up subreddit activity for {len(authors)} users (~{len(authors) * ARCTIC_MIN_INTERVAL / 60:.0f}+ min)...")
activity, act_diag = {}, []
try:
    for i, a in enumerate(authors, 1):
        data, st = arctic_get("/api/users/interactions/subreddits", {"author": a, "after": REDDIT_START, "limit": 500})
        if data is not None:
            activity[a] = parse_counts(data)
        act_diag.append({"author_hash": hash_author(a), "status": st, "n_subreddits": len(activity.get(a, {}))})
        if i % 25 == 0:
            print(f"  {i}/{len(authors)}")
except KeyboardInterrupt:
    print("Interrupted - continuing with the users looked up so far.")

# ---- 4. author table + Canadian flag ----
author_rows = []
for a, subs in activity.items():
    can = sum(v for s, v in subs.items() if s in canada_lower)
    tot = sum(subs.values())
    author_rows.append({"author": a, "sample_source": author_source[a], "total_interactions": tot,
                        "canadian_interactions": can, "canadian_share": can / tot if tot else 0,
                        "likely_canadian": can >= MIN_CANADIAN_INTERACTIONS,
                        "top_canadian_subs": ", ".join(sorted((s for s in subs if s in canada_lower), key=lambda s: -subs[s])[:5])})
author_df = pd.DataFrame(author_rows)
canadian_authors = set(author_df.loc[author_df["likely_canadian"], "author"]) if len(author_df) else set()

# share of each target subreddit's users who look Canadian - based on activity, not just the sampling source
share_rows = []
for sub in TARGET_SUBREDDITS:
    users = [a for a, subs in activity.items() if subs.get(sub.lower(), 0) > 0]
    n_can = sum(a in canadian_authors for a in users)
    share_rows.append({"target_subreddit": sub, "sampled_users_active_there": len(users),
                       "likely_canadian": n_can, "share_likely_canadian": n_can / len(users) if users else np.nan})
canada_share = pd.DataFrame(share_rows).sort_values("share_likely_canadian", ascending=False)

# ---- 5. similarity scores (author x subreddit matrix) ----
sim_target, related = pd.DataFrame(), pd.DataFrame()
if activity:
    from collections import Counter
    sub_users = Counter(s for subs in activity.values() for s in subs)
    keep = {s for s, n in sub_users.items() if n >= MIN_USERS_FOR_SIMILARITY} | {t.lower() for t in TARGET_SUBREDDITS}
    M = pd.DataFrame.from_dict({a: {s: v for s, v in subs.items() if s in keep} for a, subs in activity.items()},
                               orient="index").fillna(0.0)
    can_cols = [c for c in M.columns if c in canada_lower]
    if can_cols:
        M["CANADA (any local sub)"] = M[can_cols].sum(axis=1)
    X = np.log1p(M)
    Xn = X / np.sqrt((X ** 2).sum(axis=0)).replace(0, np.nan)
    B = (M > 0).astype(float)
    N, n_users = len(B), B.sum(axis=0)

    targets = [t.lower() for t in TARGET_SUBREDDITS if t.lower() in M.columns]
    cos = Xn.T.fillna(0) @ Xn[targets].fillna(0)                 # (all subs) x (targets)
    shared = B.T @ B[targets]
    lift = shared * N / np.outer(n_users, n_users[targets]).clip(min=1)

    long = (pd.DataFrame({"cosine_similarity": cos.stack(), "shared_users": shared.stack(),
                          "lift": pd.DataFrame(lift, index=cos.index, columns=cos.columns).stack()})
              .rename_axis(["subreddit", "target_subreddit"]).reset_index())
    long = long[long["subreddit"] != long["target_subreddit"]]
    long["users_in_sample"] = long["subreddit"].map(n_users)

    canada_cols = set(can_cols) | {"CANADA (any local sub)"}
    sim_target = (long[long["subreddit"].isin(canada_cols)]
                  .sort_values(["target_subreddit", "cosine_similarity"], ascending=[True, False]))
    related = (long[(~long["subreddit"].isin(canada_cols)) & (long["shared_users"] >= 3)]
               .sort_values(["target_subreddit", "cosine_similarity"], ascending=[True, False])
               .groupby("target_subreddit").head(30))

# ---- 6. posts by likely-Canadian users ----
canadian_posts = posts[posts["author"].isin(canadian_authors)].sort_values(
    ["subreddit", "year", "score", "num_comments"], ascending=[True, True, False, False])
reddit_posts = posts.copy()
reddit_posts["likely_canadian_author"] = reddit_posts["author"].map(
    lambda a: True if a in canadian_authors else (False if a in activity else np.nan))

def public(x):
    if x is None or len(x) == 0: return x
    x = x.copy(); x["author_hash"] = x["author"].map(hash_author); return x.drop(columns=["author"])

save_tables({"Target subs - Canadian share": canada_share,
             "Similarity to Canadian subs": sim_target,
             "Related subreddits (538-style)": related,
             "Posts by likely Canadians": public(canadian_posts),
             "Users (hashed)": public(author_df),
             "Diagnostics": pd.DataFrame(act_diag)}, "reddit_1E_canada_and_similarity")
print(canada_share.to_string(index=False))

### 1F. Describing the discourse — frequent and distinctive terms by year
A first pass at "how do people describe loneliness and isolation, and how has that changed": for each subreddit and year (2020–present), the most frequent words/two-word phrases, and the most *distinctive* ones (TF-IDF: terms used much more that year than in other years). If 1E was run, it also lists frequent terms in posts by likely-Canadian users. With `TEST_MODE` you'll only have one year, so the "distinctive" table becomes meaningful after a full run.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS

TOP_TERMS = 25
EXTRA_STOP = {"just", "like", "don", "dont", "im", "ive", "ve", "really", "know", "want", "feel", "get", "got",
              "think", "people", "would", "one", "even", "also", "much", "thing", "things", "time", "make",
              "going", "go", "https", "http", "www", "com", "amp", "x200b", "removed", "deleted", "doesn", "didn",
              "isn", "can", "cant", "ll", "re", "say", "said"}
STOP = list(ENGLISH_STOP_WORDS | EXTRA_STOP)
TOKEN = r"(?u)\b[a-zA-Z][a-zA-Z']{1,}\b"

docs = load_reddit_posts().copy()
if "body_removed" in docs:
    docs = docs[~docs["body_removed"].astype(bool)]
docs["text"] = (docs["title"].fillna("") + " " + docs["selftext"].fillna("")).str.replace(r"http\S+|&amp;|&#x200B;", " ", regex=True)
docs["year"] = docs["year"].astype(int).astype(str)

def frequent_terms(texts, n=TOP_TERMS):
    try:
        cv = CountVectorizer(stop_words=STOP, ngram_range=(1, 2), token_pattern=TOKEN, min_df=2 if len(texts) > 50 else 1)
        X = cv.fit_transform(texts)
    except ValueError:            # empty vocabulary
        return []
    sums = np.asarray(X.sum(axis=0)).ravel()
    order = sums.argsort()[::-1][:n]
    terms = cv.get_feature_names_out()
    return [(terms[i], int(sums[i])) for i in order]

freq_rows, dist_rows = [], []
for (sub, year), g in docs.groupby(["subreddit", "year"]):
    for rank, (term, n) in enumerate(frequent_terms(g["text"].tolist()), 1):
        freq_rows.append({"subreddit": sub, "year": year, "group": "all posts", "rank": rank, "term": term, "count": n, "n_posts": len(g)})

if "likely_canadian_author" in docs:
    for (sub, year), g in docs[docs["likely_canadian_author"] == True].groupby(["subreddit", "year"]):
        for rank, (term, n) in enumerate(frequent_terms(g["text"].tolist()), 1):
            freq_rows.append({"subreddit": sub, "year": year, "group": "likely Canadian authors", "rank": rank,
                              "term": term, "count": n, "n_posts": len(g)})

for sub, g in docs.groupby("subreddit"):
    by_year = g.groupby("year")["text"].apply(" ".join)
    if len(by_year) < 2:
        continue
    tv = TfidfVectorizer(stop_words=STOP, ngram_range=(1, 2), token_pattern=TOKEN, sublinear_tf=True, max_features=50000)
    X = tv.fit_transform(by_year.values).toarray()
    terms = tv.get_feature_names_out()
    for row, year in enumerate(by_year.index):
        for rank, i in enumerate(X[row].argsort()[::-1][:TOP_TERMS], 1):
            dist_rows.append({"subreddit": sub, "year": year, "rank": rank, "term": terms[i], "tfidf": round(X[row, i], 4)})

freq_df, dist_df = pd.DataFrame(freq_rows), pd.DataFrame(dist_rows)
save_tables({"Frequent terms": freq_df, "Distinctive terms by year": dist_df}, "reddit_1F_discourse_terms")

if len(freq_df):
    for sub, g in freq_df[freq_df["group"] == "all posts"].groupby("subreddit"):
        latest = g["year"].max()
        print(f"r/{sub} ({latest}) top terms:", ", ".join(g[g["year"] == latest]["term"].head(15)))

## Part 2 — Google Trends (Canada only)

All Google Trends requests are limited to Canada (`geo="CA"`), from January 2017 to today. The code retries automatically when Google throttles requests, uses today's date as the end date, and handles terms with too little Canadian search volume instead of crashing (they're listed as "no data" in the diagnostics).

How to read the numbers: Google Trends is **relative** (0–100), not search counts. 100 = the peak month *within that request*. So:
- In **2A** each term is requested separately → compare *shapes over time*, not levels between terms.
- In **2C** up to 5 terms share one request → their levels are comparable to each other (a big term can flatten small ones to 0).

Canada has a smaller search volume than the US, so niche terms are more likely to show long runs of 0. Note: pytrends is an unofficial library and its GitHub repo was archived in 2025. If Google blocks it persistently, wait a few hours or download CSVs manually from trends.google.com.

In [ ]:
from pytrends.request import TrendReq
try:
    from pytrends.exceptions import TooManyRequestsError, ResponseError
except ImportError:
    TooManyRequestsError = ResponseError = Exception

TRENDS_GEOS = {"Canada": "CA"}   # Canada only
TRENDS_TIMEFRAME = f"{START_DATE} {pd.Timestamp.now().strftime('%Y-%m-%d')}"
TRENDS_PAUSE = 6          # seconds between Google requests (plus random jitter)

pytrends = TrendReq(hl="en-US", tz=240)

def trends_retry(fn, label, retries=5, base_wait=30):
    for attempt in range(retries):
        try:
            out = fn()
            time.sleep(TRENDS_PAUSE + random.uniform(0, 3))
            return out
        except (TooManyRequestsError, ResponseError) as e:
            wait = base_wait * 2 ** attempt
            print(f"  [{label}] Google refused ({type(e).__name__}); waiting {wait}s...")
            time.sleep(wait)
    print(f"  [{label}] gave up after {retries} attempts")
    return None

def interest_over_time(terms, geo):
    def _f():
        pytrends.build_payload(terms, geo=geo, timeframe=TRENDS_TIMEFRAME)
        return pytrends.interest_over_time()
    return trends_retry(_f, f"{terms}/{geo}")

print("Google Trends helpers ready. Timeframe:", TRENDS_TIMEFRAME)

### 2B. Related queries (what else people search alongside each term)

In [ ]:
RELATED_SEEDS = ["male loneliness", "lonely", "no friends", "incel"]

rel_rows = []
for seed in RELATED_SEEDS:
    for country, geo in TRENDS_GEOS.items():
        def _f():
            pytrends.build_payload([seed], geo=geo, timeframe=TRENDS_TIMEFRAME)
            try:
                return pytrends.related_queries()
            except (IndexError, KeyError):     # pytrends crashes on low-volume terms instead of returning empty
                return {}
        res = trends_retry(_f, f"related {seed}/{geo}") or {}
        got = []
        for kind in ["top", "rising"]:
            d = (res.get(seed) or {}).get(kind)
            if d is not None and not d.empty:
                rel_rows.append(d.assign(seed=seed, country=country, kind=kind))
                got.append(f"{kind}={len(d)}")
        print(f"{seed!r:20} {country:14} {', '.join(got) or 'not enough search volume'}")

related_df = pd.concat(rel_rows, ignore_index=True) if rel_rows else pd.DataFrame(columns=["query", "value", "seed", "country", "kind"])
related_df = related_df.sort_values(["seed", "country", "kind", "value"], ascending=[True, True, True, False])
save_tables({"Related queries": related_df}, "google_2B_related_queries")
print("\n'top' = most common co-searches (scaled 0-100); 'rising' = biggest growth (% increase; very large numbers = 'Breakout').")

### 2C. Comparing terms on a shared scale (max 5 per request)

In [ ]:
COMPARE_TERMS = ["loneliness", "male loneliness", "men's mental health", "incel", "red pill"]   # 5 max

cmp_rows = []
for country, geo in TRENDS_GEOS.items():
    df = interest_over_time(COMPARE_TERMS, geo)
    if df is None or df.empty:
        print(f"{country}: no data"); continue
    df = df.drop(columns=["isPartial"], errors="ignore").reset_index().melt("date", var_name="term", value_name="interest")
    cmp_rows.append(df.assign(country=country))

compare_df = pd.concat(cmp_rows, ignore_index=True) if cmp_rows else pd.DataFrame()
save_tables({"Shared-scale comparison": compare_df}, "google_2C_term_comparison")

if len(compare_df):
    countries = compare_df["country"].unique()
    fig, axes = plt.subplots(len(countries), 1, figsize=(13, 4 * len(countries)), sharex=True, squeeze=False)
    for ax, country in zip(axes[:, 0], countries):
        for term, g in compare_df[compare_df["country"] == country].groupby("term"):
            ax.plot(g["date"], g["interest"], label=term)
        ax.set_title(f"{country}: terms on one shared 0-100 scale"); ax.legend(); ax.grid(alpha=.3)
    plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, "google_2C_term_comparison.png"), dpi=150); plt.show()

## Notes for the write-up
- **Reddit source:** Arctic Shift (posts, comments, subreddit and user activity), a free, volunteer-run archive of public Reddit data with no uptime guarantee. Check each `Diagnostics` sheet so failed requests are never mistaken for real zeros.
- **Periods:** 1A is monthly from 2017; 1B, 1C, 1E and 1F are yearly from 2020; Google Trends is monthly from 2017, Canada only. The current year is partial.
- **Popularity:** every Reddit table is sorted by score (then comment count). Scores of posts under ~36 h old are provisional.
- **Canada:** Reddit has no location data; "likely Canadian" is inferred from activity in Canadian local subreddits (see 1E caveats). Google Trends gives true national and provincial figures.
- **Privacy:** exported files contain hashed usernames only. `reddit_posts_raw.pkl` keeps raw usernames for the pipeline — don't share it.

In [2]:
import pandas as pd

In [13]:
data1 = pd.read_csv("C:/Users/vesta/Downloads/project_outputs/reddit_1B_posts_by_popularity__All_matching_posts.csv")

In [14]:
data1.head()

,subreddit,year,date,score,num_comments,title,selftext,keyword_hits,flair,post_id,permalink,body_removed,matched_search_keywords,rank_in_year,partial_year,author_hash
0,AskMen,2020,2020-12-29 17:50:06,26011,1848,My best friend lost a bunch of weight and now ...,"Before losing weight, he treated women well an...",alone,NaN,kmj45u,https://www.reddit.com/r/AskMen/comments/kmj45u/,False,alone,1,False,4fd4eeda5c18
1,AskMen,2020,2020-06-06 00:45:37,21881,2106,"To all my single fellas living alone, how are ...",I live on my own and am single. Has been quite...,alone; lonely,NaN,gxgx5c,https://www.reddit.com/r/AskMen/comments/gxgx5c/,False,alone,2,False,084b05a0f548
2,AskMen,2020,2020-08-30 11:49:36,12558,1574,How have your friendships been this year?,"For me, it made me realize I have no friends. ...",lonely; no friends,NaN,ijbgma,https://www.reddit.com/r/AskMen/comments/ijbgma/,False,no friends,3,False,5b0ca1a50a17
3,AskMen,2020,2020-08-26 10:17:55,11521,1047,"Men, how are you all so gorgeous and adorable?","If anything, this sub has taught me to: \n\n- ...",NaN,NaN,igwadn,https://www.reddit.com/r/AskMen/comments/igwadn/,False,no friends,4,False,ce0a28970ff3
4,AskMen,2020,2020-12-25 12:37:59,9729,1899,Fellow men of Reddit who are spending the holi...,NaN,alone,NaN,kjyeds,https://www.reddit.com/r/AskMen/comments/kjyeds/,False,alone,5,False,29e0491b3a57


In [15]:
data1 = data1.sort_values(by = "year")

In [16]:
data1.head()

,subreddit,year,date,score,num_comments,title,selftext,keyword_hits,flair,post_id,permalink,body_removed,matched_search_keywords,rank_in_year,partial_year,author_hash
19,AskMen,2020,2020-08-21 02:41:03,779,285,"Do you ever have days where you feel think ""I'...","For any lady readers, any insight you have wou...",NaN,NaN,idoibr,https://www.reddit.com/r/AskMen/comments/idoibr/,False,no friends,20,False,[deleted]
18,AskMen,2020,2020-05-16 05:23:52,822,139,How do you start dating with no experience?,I'm 22 and had always had trouble talking to p...,alone,NaN,gkp8uh,https://www.reddit.com/r/AskMen/comments/gkp8uh/,False,alone,19,False,e48a2aa71332
17,AskMen,2020,2020-03-31 21:11:25,895,381,How was college for you? Can you relate to not...,I'm not sure if you're familiar with the actre...,lonely,NaN,fslfj1,https://www.reddit.com/r/AskMen/comments/fslfj1/,False,lonely,18,False,4d87d15dddee
16,AskMen,2020,2020-01-03 04:08:15,1235,416,How come some people meet their SO easily and ...,Recently two of my friends became single. We a...,NaN,NaN,ej9ybd,https://www.reddit.com/r/AskMen/comments/ej9ybd/,False,no friends,17,False,631f35500a9e
15,AskMen,2020,2020-06-28 17:58:07,1274,288,"Men, how do you move on knowing she already has?",Backstory: I (27m)was recently dumped from a 4...,alone; no friends,NaN,hhizpw,https://www.reddit.com/r/AskMen/comments/hhizpw/,False,alone,16,False,8f82b76dada0


In [17]:
data1 = data1.sort_values(by = "year", ascending = False)

In [18]:
data1.head()

,subreddit,year,date,score,num_comments,title,selftext,keyword_hits,flair,post_id,permalink,body_removed,matched_search_keywords,rank_in_year,partial_year,author_hash
466007,lonely,2026,2026-10-04 04:25:54,0,0,Im in my lonliest phase,[removed],NaN,NaN,1wx6l2a,https://www.reddit.com/r/lonely/comments/1wx6l2a/,True,(all posts),33171,True,2b6a94ac345c
466039,lonely,2026,2026-10-06 17:06:14,0,0,"Reddit keeps deleting this, but I'm going to c...",[removed],NaN,Venting,1wz82ct,https://www.reddit.com/r/lonely/comments/1wz82ct/,True,(all posts),33203,True,9a367c35b4d0
98477,ForeverAlone,2026,2026-08-28 09:48:22,0,0,He moved on when I showed him that I don’t car...,"This guy was so invested in me, I was extremel...",NaN,Vent,1w0m2d6,https://www.reddit.com/r/ForeverAlone/comments...,False,(all posts),5097,True,63695558c28b
466013,lonely,2026,2026-10-04 17:26:20,0,0,22F | Hoping to meet some wonderful new people,[removed],NaN,NaN,1wxl9nu,https://www.reddit.com/r/lonely/comments/1wxl9nu/,True,(all posts),33177,True,443c35806770
466045,lonely,2026,2026-10-06 22:50:56,0,0,24F So hard to just keep going,[removed],NaN,NaN,1wzgqjl,https://www.reddit.com/r/lonely/comments/1wzgqjl/,True,(all posts),33209,True,0c5cb71c7108


In [19]:
data2 = data1[data1["subreddit"] == "lonely"]

In [20]:
data2.head()

,subreddit,year,date,score,num_comments,title,selftext,keyword_hits,flair,post_id,permalink,body_removed,matched_search_keywords,rank_in_year,partial_year,author_hash
466007,lonely,2026,2026-10-04 04:25:54,0,0,Im in my lonliest phase,[removed],NaN,NaN,1wx6l2a,https://www.reddit.com/r/lonely/comments/1wx6l2a/,True,(all posts),33171,True,2b6a94ac345c
466039,lonely,2026,2026-10-06 17:06:14,0,0,"Reddit keeps deleting this, but I'm going to c...",[removed],NaN,Venting,1wz82ct,https://www.reddit.com/r/lonely/comments/1wz82ct/,True,(all posts),33203,True,9a367c35b4d0
466013,lonely,2026,2026-10-04 17:26:20,0,0,22F | Hoping to meet some wonderful new people,[removed],NaN,NaN,1wxl9nu,https://www.reddit.com/r/lonely/comments/1wxl9nu/,True,(all posts),33177,True,443c35806770
466045,lonely,2026,2026-10-06 22:50:56,0,0,24F So hard to just keep going,[removed],NaN,NaN,1wzgqjl,https://www.reddit.com/r/lonely/comments/1wzgqjl/,True,(all posts),33209,True,0c5cb71c7108
466030,lonely,2026,2026-10-05 23:28:01,0,0,How do people deal with being lonely?,I’m over a month into college and I still have...,alone; disconnected; lonely,Discussion,1wynkch,https://www.reddit.com/r/lonely/comments/1wynkch/,False,(all posts),33194,True,4e0cbc715f73


In [21]:
len(data1), len(data2)

(466061, 367568)

In [22]:
data3 = data1[data1["selftext"]!= "[removed]"]

In [25]:
data3.head()

,subreddit,year,date,score,num_comments,title,selftext,keyword_hits,flair,post_id,permalink,body_removed,matched_search_keywords,rank_in_year,partial_year,author_hash
98477,ForeverAlone,2026,2026-08-28 09:48:22,0,0,He moved on when I showed him that I don’t car...,"This guy was so invested in me, I was extremel...",NaN,Vent,1w0m2d6,https://www.reddit.com/r/ForeverAlone/comments...,False,(all posts),5097,True,63695558c28b
98492,ForeverAlone,2026,2026-10-05 17:46:48,0,0,i’m stuck in online dating hell.,it feels like online dating is the only way no...,NaN,Vent,1wyf4cf,https://www.reddit.com/r/ForeverAlone/comments...,False,(all posts),5112,True,b8bd33683cfe
466030,lonely,2026,2026-10-05 23:28:01,0,0,How do people deal with being lonely?,I’m over a month into college and I still have...,alone; disconnected; lonely,Discussion,1wynkch,https://www.reddit.com/r/lonely/comments/1wynkch/,False,(all posts),33194,True,4e0cbc715f73
466044,lonely,2026,2026-10-06 20:27:32,0,0,My therapist disappointed me,I can't afford a therapist because therapy is ...,NaN,NaN,1wzdbuc,https://www.reddit.com/r/lonely/comments/1wzdbuc/,False,(all posts),33208,True,9235db06d886
466043,lonely,2026,2026-10-06 18:05:04,0,0,"My feelings, my thoughts","I just need somewhere to vent, to be heard by ...",alone; no friends,TW: Vent/Life,1wz9lnq,https://www.reddit.com/r/lonely/comments/1wz9lnq/,False,(all posts),33207,True,a49a40cb9998


In [26]:
len(data3)

374868

In [28]:
data4 = data1[(data1["title"].str.contains("F ", na = False))|(data1["selftext"].str.contains("F ", na = False))]

In [29]:
len(data4)

9996